# Ноутбук для решения задач из 4 модуля



In [ ]:
!pip install langchain langchain-community huggingface_hub langchain-openai openai faiss-gpu langchainhub -q

## Если используете ключ от OpenAI, запустите эту ячейку 👇







In [ ]:
from langchain_openai import ChatOpenAI
import os
from getpass import getpass


# os.environ['OPENAI_API_KEY'] = "Введите ваш OpenAI API ключ"
os.environ['OPENAI_API_KEY'] = getpass(prompt='Введите ваш OpenAI API ключ')

# Инициализируем языковую модель
llm = ChatOpenAI(temperature=0.0)

## Если используете ключ из курса, запустите эти ячейки 👇


In [ ]:
!wget https://raw.githubusercontent.com/a-milenkin/LLM_practical_course/main/notebooks/utils.py

--2024-05-08 19:19:05--  https://raw.githubusercontent.com/a-milenkin/LLM_practical_course/main/notebooks/utils.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 11184 (11K) [text/plain]
Saving to: ‘utils.py.1’

utils.py.1          100%[===================>]  10.92K  --.-KB/s    in 0s      

2024-05-08 19:19:05 (87.8 MB/s) - ‘utils.py.1’ saved [11184/11184]



In [ ]:
from utils import ChatOpenAI
from getpass import getpass

#course_api_key= "Введите ваш API ключ с курса"
course_api_key = getpass(prompt='Введите API ключ')

# Инициализируем языковую модель
llm = ChatOpenAI(temperature=0.0, course_api_key=course_api_key)

Введите API ключ··········


## Задание из стэпа 6.🪛 Openai tools Agent + RAG 🦞

Распарсив веб страницу, создадим базу знаний

In [ ]:
from langchain.document_loaders import WebBaseLoader

In [ ]:
loader = WebBaseLoader("https://ru.wikipedia.org/wiki/%D0%93%D0%B0%D0%BD%D0%BD%D0%B8%D0%B1%D0%B0%D0%BB,_%D0%90%D0%B1%D1%80%D0%B0%D0%BC_%D0%9F%D0%B5%D1%82%D1%80%D0%BE%D0%B2%D0%B8%D1%87")
data = loader.load()

Засплитим данные, сформируем эмбединги и создадим ретривер


In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import CharacterTextSplitter
from utils import OpenAIEmbeddings

text_splitter = CharacterTextSplitter(chunk_size=2000, chunk_overlap=100)
texts = text_splitter.split_documents(data)
embeddings = OpenAIEmbeddings(course_api_key=course_api_key)
db_embed = FAISS.from_documents(texts, embeddings)

retriever = db_embed.as_retriever()

Инструменты для агента

In [ ]:
from langchain.tools.retriever import create_retriever_tool

tool = create_retriever_tool(
    retriever, # наш ретривер
    "search_web", # имя инструмента
    "Searches and returns data from page", # описание инструмента подается в ЛЛМ
)
tools = [tool]

In [ ]:
from langchain import hub

prompt = hub.pull("hwchase17/openai-tools-agent")

prompt

ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], input_types={'chat_history': typing.List[typing.Union[langchain_core.messages.ai.AIMessage, langchain_core.messages.human.HumanMessage, langchain_core.messages.chat.ChatMessage, langchain_core.messages.system.SystemMessage, langchain_core.messages.function.FunctionMessage, langchain_core.messages.tool.ToolMessage]], 'agent_scratchpad': typing.List[typing.Union[langchain_core.messages.ai.AIMessage, langchain_core.messages.human.HumanMessage, langchain_core.messages.chat.ChatMessage, langchain_core.messages.system.SystemMessage, langchain_core.messages.function.FunctionMessage, langchain_core.messages.tool.ToolMessage]]}, metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'openai-tools-agent', 'lc_hub_commit_hash': 'c18672812789a3b9697656dd539edf0120285dcae36396d0b548ae42a4ed66f5'}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], template='You are a helpful assistant')), MessagesPlacehold

In [ ]:
from langchain_core.prompts.prompt import PromptTemplate
from langchain_core.prompts.chat import MessagesPlaceholder
from langchain_core.prompts.chat import HumanMessagePromptTemplate

In [ ]:
prompt.messages = [SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], template='brief short answer, very important: less than 5 words in final answer')), MessagesPlaceholder(variable_name='chat_history', optional=True), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], template='{input}')), MessagesPlaceholder(variable_name='agent_scratchpad')]

In [ ]:
prompt

ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], input_types={'chat_history': typing.List[typing.Union[langchain_core.messages.ai.AIMessage, langchain_core.messages.human.HumanMessage, langchain_core.messages.chat.ChatMessage, langchain_core.messages.system.SystemMessage, langchain_core.messages.function.FunctionMessage, langchain_core.messages.tool.ToolMessage]], 'agent_scratchpad': typing.List[typing.Union[langchain_core.messages.ai.AIMessage, langchain_core.messages.human.HumanMessage, langchain_core.messages.chat.ChatMessage, langchain_core.messages.system.SystemMessage, langchain_core.messages.function.FunctionMessage, langchain_core.messages.tool.ToolMessage]]}, metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'openai-tools-agent', 'lc_hub_commit_hash': 'c18672812789a3b9697656dd539edf0120285dcae36396d0b548ae42a4ed66f5'}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], template='brief short answer, very important: less than 5 w

Подключаем агента

In [ ]:
from langchain.agents import create_openai_tools_agent
from langchain.agents.agent import AgentExecutor

agent = create_openai_tools_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools)

In [ ]:
import pandas as pd
from tqdm import tqdm

In [ ]:
# Загружаем датасет с вопросами про Ганнибала
df = pd.read_csv("https://stepik.org/media/attachments/lesson/1107866/gannibal.csv")

In [ ]:
agent_executor

AgentExecutor(agent=RunnableMultiActionAgent(runnable=RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_to_openai_tool_messages(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], input_types={'chat_history': typing.List[typing.Union[langchain_core.messages.ai.AIMessage, langchain_core.messages.human.HumanMessage, langchain_core.messages.chat.ChatMessage, langchain_core.messages.system.SystemMessage, langchain_core.messages.function.FunctionMessage, langchain_core.messages.tool.ToolMessage]], 'agent_scratchpad': typing.List[typing.Union[langchain_core.messages.ai.AIMessage, langchain_core.messages.human.HumanMessage, langchain_core.messages.chat.ChatMessage, langchain_core.messages.system.SystemMessage, langchain_core.messages.function.FunctionMessage, langchain_core.messages.tool.ToolMessage]]}, metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'openai-tools-agent', 'lc_hub_commit_hash': 'c18672812789a3b9697656

In [ ]:
answers = [] # Список, где будем хранить ответы модели

for text_input in tqdm(df['question']):
    answer = agent_executor.invoke({'input': text_input})
    answers.append(answer['output']) # Добавляем ответ в список
    #break # Для отладки. Уберите, когда убедитесь, что на одном примере работает

100%|██████████| 10/10 [00:27<00:00,  2.72s/it]


In [ ]:
answers

['Евдокия Андреевна Диопер',
 'Да',
 '84 года',
 '1727',
 'В голову.',
 '100 рублей в год.',
 'Со Швецией.',
 'генерал-аншеф',
 'Христина-Регина фон Шёберг',
 '4 детей']

In [ ]:
df['answer'] = answers # Создаём новый столбец из ответов модели

In [ ]:
df.to_csv('step6_solution.csv', index=False) # Сохраняем файл, отправляем на Stepik, получаем баллы :)